# 09 — Surface-Code Foundations: One Logical Qubit on a Patch

## Read before you begin

- Fowler et al., [*Surface codes*](https://arxiv.org/abs/1208.0928), sections **III**, **VI**, and **VII**. Extract data/measurement roles, boundaries, and repeated measurement.
- Revisit notebook 08’s difference among physical errors, syndromes, and corrections.

## Learning objectives

- Describe storage of one logical qubit while local checks are repeatedly measured.
- Identify data, ancillas, checks, strings, raw outcomes, detectors, and decoder outputs.
- Follow one check before a complete round.

## Prerequisites and how to use this notebook

Know stabilizers, CSS codes, and Steane code. A rotated distance-$d$ patch has $n=d^2$ **data** qubits and $k=1$; temporary ancillas are excluded from $[[n,k,d]]$.

Run cells in order. Pause at each prediction and make a claim before revealing the output.

**By the end:** you will be ready to read the detector graph and matching decoder in notebook 10.

## 1. The experiment

We store **one logical qubit** in a two-dimensional data-qubit patch while repeatedly measuring small local stabilizers using resettable ancillas. Checks reveal parity changes, not the encoded logical state.

- **Data qubits** hold the encoded state.
- **Measurement ancillas** are reset, coupled locally, measured, and reused.
- A **raw syndrome outcome** is one check result in one round.
- A **detector event** is an unexpected parity/change formed from raw outcomes, often across rounds.
- A **decoder output** is a proposed Pauli-frame update. A Pauli frame tracks known corrections in software; it does not make arbitrary states valid code states.

## 2. Physical patch geometry

The roles below are inferred from **Stim’s generated-circuit convention**: final `M` targets are data, repeated `MR` targets are ancillas, and Hadamard-targeted ancillas are $X$ checks. That is useful here, not a universal QEC convention.

In [ ]:
import stim, matplotlib.pyplot as plt

def layout(distance, rounds=1):
    circuit=stim.Circuit.generated("surface_code:rotated_memory_z", distance=distance, rounds=rounds)
    coords=circuit.get_final_qubit_coordinates(); data=set(); anc=set(); h=set()
    for instruction in circuit:
        targets={int(t.value) for t in instruction.targets_copy()}
        if instruction.name=="M": data |= targets
        elif instruction.name=="MR": anc |= targets
        elif instruction.name=="H": h |= targets
    return circuit,coords,data,anc & h,anc-h

circuit_d3,coords,data_qubits,x_ancillas,z_ancillas=layout(3)
print(f"[[{len(data_qubits)}, 1, 3]]: {len(data_qubits)} data qubits and {len(x_ancillas)+len(z_ancillas)} check ancillas")

fig,ax=plt.subplots(figsize=(5,5),facecolor="white")
for qs,color,label,marker in [(data_qubits,"white","data qubit","o"),(x_ancillas,"#E57373","X-check ancilla","s"),(z_ancillas,"#64B5F6","Z-check ancilla","s")]:
    ax.scatter([coords[q][0] for q in qs],[coords[q][1] for q in qs],s=260,c=color,marker=marker,edgecolors="#263238",label=label)
ax.set(title="Distance-3 rotated patch: roles only",aspect="equal",facecolor="white");ax.invert_yaxis();ax.axis("off");ax.legend();plt.show()

**Figure caption.** White circles are data qubits; red squares measure $X$-type checks; blue squares measure $Z$-type checks. This view intentionally has no supports, logical strings, or detector information.

## 3. Local check supports

Each ancilla couples only to neighboring data. The next cell extracts supports from the generated circuit and highlights one check. Interior checks have weight four; boundary checks can have weight two.

In [ ]:
def supports(circuit,data,ancillas):
    result={a:set() for a in ancillas}
    for instruction in circuit:
        if instruction.name=="CX":
            ts=[int(t.value) for t in instruction.targets_copy()]
            for a,b in zip(ts[::2],ts[1::2]):
                pair={a,b}; aa=pair & ancillas; dd=pair & data
                if len(aa)==len(dd)==1: result[next(iter(aa))].add(next(iter(dd)))
    return result

check_supports=supports(circuit_d3,data_qubits,x_ancillas|z_ancillas)
ancilla=sorted(x_ancillas)[0]; touched=check_supports[ancilla]
print(f"X-check ancilla {ancilla} measures X on data qubits {sorted(touched)}")
fig,ax=plt.subplots(figsize=(5,5),facecolor="white")
for q in data_qubits:
    x,y=coords[q][:2]; ax.scatter(x,y,s=300,c="#FFF3CD" if q in touched else "white",edgecolors="#263238");ax.text(x,y,str(q),ha="center",va="center")
x,y=coords[ancilla][:2];ax.scatter(x,y,s=300,marker="s",c="#E57373",edgecolors="#263238")
for q in touched: ax.plot([x,coords[q][0]],[y,coords[q][1]],color="#D32F2F",lw=2)
ax.set(title="One selected X-check support",aspect="equal",facecolor="white");ax.invert_yaxis();ax.axis("off");plt.show()

**Figure caption.** The red square is one selected $X$-check ancilla. Pale-yellow circles are exactly its data support; red lines are local couplings. Stim red/green/blue mean $X/Y/Z$ **Pauli support**, not errors.

### Checkpoint

An $X$ error anticommutes with nearby $Z$ checks; a $Z$ error anticommutes with nearby $X$ checks. This is the same CSS cross-checking rule as in Steane code, now made local.

## 4. One ancilla-based stabilizer measurement

First see a teaching-sized $Z\otimes Z$ measurement: reset an ancilla, copy parity onto it with CNOTs, then measure it. The data are not measured.

In [ ]:
from IPython.display import display
one_check=stim.Circuit("R 2\nCX 0 2 1 2\nM 2")
display(one_check.diagram("timeline-svg"))

**Figure caption.** Lines 0 and 1 are data; line 2 is the ancilla. `R` resets to $|0\rangle$, `CX` couples data parity to the ancilla, and `M` reads it in the $Z$ basis. $X$ checks use `H` basis changes. `MR` means measure then reset for reuse.

## 5. One full syndrome-extraction round

`TICK` is only a timing separator. `DETECTOR` defines a known parity of measurement-record bits; `OBSERVABLE_INCLUDE` identifies record bits whose parity represents a logical observable. Neither applies a correction. The diagram is only the four CNOT layers, not a raw full-circuit dump.

In [ ]:
display(circuit_d3.diagram("timeslice-svg",tick=range(2,6),rows=1))

**Figure caption.** Each panel is one CNOT layer. Following an ancilla across panels reveals its local support and the schedule avoids simultaneous gate conflicts.

## 6. Repeated rounds and detection events

We force one $X$ error after initialization, then sample three rounds. The printed Boolean positions are **detector events**, not raw ancilla outcomes.

In [ ]:
import numpy as np
multi=stim.Circuit.generated("surface_code:rotated_memory_z",distance=3,rounds=3)
i=next(i for i,v in enumerate(multi) if v.name=="R"); faulty=multi.copy(); faulty.insert(i+1,stim.CircuitInstruction("X_ERROR",[min(data_qubits)],[1.0]))
events=faulty.compile_detector_sampler().sample(shots=1)[0]
print("fired detector indices:",np.flatnonzero(events).tolist())
print("raw measurement bits:",faulty.num_measurements," detector-event bits:",faulty.num_detectors)

### Checkpoint

Physical errors are faults; syndrome outcomes are individual check measurements; detection events are selected parity changes. A decoder proposes a **correction chain**. If physical and correction chains differ by a logical string rather than stabilizers, the result is a logical error.

## 7. First decoding and logical failure

Matching receives detector events and predicts observable flips. Comparing the prediction with Stim’s sampled observable flip estimates decoder failure. This is a preview of notebook 10.

In [ ]:
import pymatching
noisy=stim.Circuit.generated("surface_code:rotated_memory_z",distance=3,rounds=3,after_clifford_depolarization=.005)
matcher=pymatching.Matching.from_detector_error_model(noisy.detector_error_model(decompose_errors=True))
det,actual=noisy.compile_detector_sampler().sample(shots=2000,separate_observables=True)
predicted=matcher.decode_batch(det); failures=np.count_nonzero(np.any(predicted!=actual,axis=1))
print(f"{failures}/2000 decoder failures; estimated logical failure rate {failures/2000:.4f}")

## 8. Distance is a geometric resource

The shortest logical string crosses appropriate opposite boundaries and has length $d$. The local check pattern remains the same while data-qubit cost grows as $d^2$.

In [ ]:
ds=[3,5,7,9]; ns=[]
for d in ds:
    _,_,data,xs,zs=layout(d);ns.append(len(data));print(f"d={d}: [[{len(data)},1,{d}]], {len(xs)+len(zs)} checks")
fig,ax=plt.subplots(figsize=(5,3.2),facecolor="white");ax.plot(ds,ns,"o-",color="#1565C0",lw=2);ax.set(xlabel="distance d",ylabel="data qubits n=d²",title="Protection costs geometric area",facecolor="white");ax.grid(alpha=.25);plt.show()

**Figure caption.** Each point is a rotated patch. The vertical axis counts data qubits only; check ancillas are excluded from $[[n,k,d]]$.

## Can you now explain? — Exercises

1. What is stored, what is measured repeatedly, and what remains hidden?
2. Why does an $X$ error affect $Z$ checks?
3. Distinguish an ancilla outcome, detector event, and decoder output.
4. When can two correction chains both be acceptable, and when is their difference logical?
5. Why is distance the shortest boundary-to-boundary logical string?

**Bridge.** Notebook 10 makes the decoder graph explicit.